# Tool Testing Notebook

This notebook lets you test each of the 6 scientific database tools individually,
before running the full agent loop.

**Tools covered:**
1. `search_pubmed` — PubMed literature search
2. `search_trials` — ClinicalTrials.gov v2
3. `lookup_protein` — UniProt protein/function lookup
4. `query_drug_targets` — Open Targets drug–gene associations
5. `lookup_drug` — ChEMBL compound + mechanism
6. `search_genes` — NCBI Gene database

**Run the full agent** at the bottom of this notebook.

> **Prerequisites:** set `GROQ_API_KEY` and optionally `NCBI_EMAIL` in your `.env` file.

In [ ]:
# Setup — run this cell first
import sys, os, json
from pathlib import Path

# Add project root to path
ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
sys.path.insert(0, str(ROOT))

# Load .env (optional — or set env vars manually)
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
    print('✅ .env loaded')
except ImportError:
    print('dotenv not installed — set GROQ_API_KEY manually if needed')

def pretty(result: dict):
    """Pretty-print a tool result dict."""
    print(json.dumps(result, indent=2, default=str)[:3000])

print(f'Python: {sys.version}')
print(f'GROQ_API_KEY set: {bool(os.environ.get("GROQ_API_KEY"))}')
print(f'NCBI_EMAIL: {os.environ.get("NCBI_EMAIL", "not set — will use default")}')

---
## 1. PubMed — `search_pubmed`

Searches peer-reviewed literature via NCBI Entrez. Free, no API key needed (though
setting `NCBI_EMAIL` gives you higher rate limits).

In [ ]:
from src.tools.pubmed import search_pubmed

result = search_pubmed(
    query="KRAS G12C inhibitor sotorasib clinical trial",
    max_results=3,
    sort="relevance",
)
pretty(result)

In [ ]:
# Quick summary of papers found
for p in result.get('papers', []):
    print(f"PMID {p['pmid']} ({p['year']}) — {p['title'][:80]}")
    print(f"  Journal: {p['journal']}")
    print(f"  Abstract preview: {p.get('abstract','')[:200]}...")
    print()

---
## 2. ClinicalTrials.gov — `search_trials`

Queries the ClinicalTrials.gov v2 API. Supports filtering by status and phase.

In [ ]:
from src.tools.clinical_trials import search_trials

result = search_trials(
    query="KRAS G12C inhibitor",
    max_results=5,
    status_filter="RECRUITING",
    phase_filter="PHASE3",
)
pretty(result)

In [ ]:
# Summary table of trials found
print(f"{'NCT ID':<15} {'Phase':<8} {'Status':<18} {'Title'[:50]}")
print('-' * 80)
for t in result.get('trials', []):
    print(f"{t['nct_id']:<15} {str(t.get('phase','?')):<8} {t.get('status','?'):<18} {t['title'][:50]}")

---
## 3. UniProt — `lookup_protein`

Retrieves protein function, disease associations, subcellular location, and
pathways from the UniProt REST API.

In [ ]:
from src.tools.uniprot import lookup_protein

result = lookup_protein(
    query="KRAS",
    organism="Homo sapiens",
    max_results=1,
)
pretty(result)

In [ ]:
# Parse key fields
for p in result.get('proteins', []):
    print(f"Accession : {p['accession']}")
    print(f"Name      : {p['protein_name']}")
    print(f"Gene      : {p['gene_name']}")
    print(f"Function  : {p.get('function','')[:400]}")
    print(f"Diseases  : {p.get('disease_associations', [])}")
    print(f"URL       : {p['url']}")

---
## 4. Open Targets — `query_drug_targets`

Uses the Open Targets GraphQL API to find drugs associated with a given gene target,
along with their clinical stage and indications.

In [ ]:
from src.tools.opentargets import query_drug_targets

result = query_drug_targets(
    target_gene="KRAS",
    max_drugs=5,
    min_phase=2,
)
pretty(result)

In [ ]:
# Summary table
print(f"{'Drug':<25} {'Phase':<8} {'MoA'[:40]}")
print('-' * 75)
for d in result.get('drugs', []):
    print(f"{d['drug_name']:<25} {str(d.get('clinical_phase','?')):<8} {d.get('mechanism_of_action','?')[:40]}")

---
## 5. ChEMBL — `lookup_drug`

Looks up a compound by name in ChEMBL. Returns molecule type, clinical phase,
approval status, SMILES, and mechanisms of action.

In [ ]:
from src.tools.chembl import lookup_drug

result = lookup_drug(
    drug_name="sotorasib",
    max_results=2,
)
pretty(result)

In [ ]:
# Parse key fields
for d in result.get('drugs', []):
    print(f"ChEMBL ID    : {d['chembl_id']}")
    print(f"Name         : {d['name']}")
    print(f"Type         : {d['molecule_type']}")
    print(f"Max phase    : {d['max_clinical_phase']}")
    print(f"Approved     : {d['is_approved']}")
    print(f"Mol. weight  : {d['molecular_weight']}")
    print(f"Mechanisms   :")
    for m in d.get('mechanisms_of_action', []):
        print(f"  - {m['mechanism']} ({m['action_type']}) → {m['target_name']}")
    print(f"URL          : {d['url']}")

---
## 6. NCBI Gene — `search_genes`

Searches the NCBI Gene database for gene symbols, chromosomal location,
gene type, aliases, and OMIM disease links.

In [ ]:
from src.tools.ncbi_gene import search_genes

result = search_genes(
    query="KRAS",
    organism="Homo sapiens",
    max_results=1,
)
pretty(result)

In [ ]:
for g in result.get('genes', []):
    print(f"Symbol   : {g['symbol']}")
    print(f"Full name: {g['full_name']}")
    print(f"Type     : {g['gene_type']}")
    print(f"Location : chr{g['chromosome']}, {g['chromosomal_location']}")
    print(f"Aliases  : {g['aliases']}")
    print(f"OMIM IDs : {g['omim_ids']}")
    print(f"Summary  : {g.get('summary','')[:300]}...")
    print(f"URL      : {g['ncbi_url']}")

---
## 7. Tool Dispatcher — `call_tool`

This is the function the agent uses internally. It dispatches by name and returns a JSON string.

In [ ]:
from src.tools import call_tool, TOOL_REGISTRY

print('Registered tools:', list(TOOL_REGISTRY.keys()))

# Test dispatcher
result_json = call_tool('search_genes', {'query': 'TP53', 'max_results': 1})
import json
print(json.dumps(json.loads(result_json), indent=2)[:1000])

In [ ]:
# Test error handling — unknown tool
result_json = call_tool('nonexistent_tool', {})
print(result_json)

---
## 8. Full Agent Run

Now run the complete ReAct loop. The agent will call multiple tools and synthesize
a final answer. Requires `GROQ_API_KEY` to be set.

In [ ]:
from src.agent import BiotechResearchAgent

# Check API key
if not os.environ.get('GROQ_API_KEY'):
    print('❌ GROQ_API_KEY not set. Set it in .env or: os.environ["GROQ_API_KEY"] = "your_key"')
else:
    print('✅ GROQ_API_KEY is set')

In [ ]:
# Initialise agent
agent = BiotechResearchAgent(
    model="llama3-70b-8192",
    max_iterations=8,
    temperature=0.1,
)
print(f'Agent ready: model={agent.model}, max_iter={agent.max_iterations}')

In [ ]:
# Run a research question
question = "What is the mechanism of sotorasib and what phase 3 trials are running?"

import time
t0 = time.time()
result = agent.run(question)
elapsed = time.time() - t0

print(f'\n⏱ Completed in {elapsed:.1f}s')
print(f'Iterations: {result.iterations}')
print(f'Tool calls: {len(result.tool_calls)}')
print(f'Stop reason: {result.stopped_reason}')
print('\n' + '='*60 + '\nANSWER\n' + '='*60)
print(result.answer)

In [ ]:
# Inspect tool call trace
print(f'\n{"="*60}\nTOOL TRACE\n{"="*60}')
for tc in result.tool_calls:
    print(f'\n[Iter {tc.iteration}] {tc.tool_name}({tc.arguments})')
    try:
        preview = json.loads(tc.result)
        # Just show keys and counts
        for k, v in preview.items():
            if isinstance(v, list):
                print(f'  {k}: [{len(v)} items]')
            elif isinstance(v, str):
                print(f'  {k}: "{v[:80]}"')
            else:
                print(f'  {k}: {v}')
    except Exception:
        print(f'  (raw): {tc.result[:200]}')

In [ ]:
# Generate a markdown report
from src.report_generator import generate_report, save_report

report_md = generate_report(result)
output_path = save_report(report_md, ROOT / 'outputs' / 'report.md')
print(f'Report saved to: {output_path}')
print('\nFirst 1000 chars:')
print(report_md[:1000])

---
## 9. Config-driven Agent

You can also load the agent from `config.yaml` instead of passing arguments directly.

In [ ]:
# Load from config.yaml
agent_from_config = BiotechResearchAgent.from_config(str(ROOT / 'config.yaml'))
print(f'Loaded: model={agent_from_config.model}, max_iter={agent_from_config.max_iterations}')

---
## 10. Try Your Own Question

Edit the question below and run to see what the agent does.

In [ ]:
my_question = "What drugs target EGFR and are approved for non-small cell lung cancer?"

result2 = agent.run(my_question)
print(result2.answer)